# Project 08: Core NLP Sentiment Classification (Bridge + Observatory Pro)

Train a neural sentiment classifier on IMDb reviews and monitor the run with
adaptive warnings, tiered storage, and dashboard export.


## Dataset Source and Download Instructions

- Dataset: IMDb Large Movie Review Dataset
- Official source: https://ai.stanford.edu/~amaas/data/sentiment/
- License: see Stanford dataset terms
- Auto-download in this notebook: Hugging Face `datasets.load_dataset("imdb")`
- Manual fallback:
  1. Download the tarball from Stanford.
  2. Extract under `./data/imdb/aclImdb/`.
  3. Adapt loading cell to local files.


In [ ]:
%pip -q install "neurogebra[logging]" torch datasets scikit-learn matplotlib


In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from torch.utils.data import DataLoader, TensorDataset

from neurogebra import Expression
from neurogebra.bridges.pytorch_bridge import to_pytorch
from neurogebra.logging.adaptive import AdaptiveLogger, AnomalyConfig
from neurogebra.logging.dashboard import DashboardExporter
from neurogebra.logging.health_warnings import AutoHealthWarnings
from neurogebra.logging.logger import LogLevel, TrainingLogger
from neurogebra.logging.tiered_storage import TieredStorage

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
imdb = load_dataset("imdb")

train_text = imdb["train"]["text"][:6000]
train_label = np.array(imdb["train"]["label"][:6000], dtype=np.float32)
test_text = imdb["test"]["text"][:2000]
test_label = np.array(imdb["test"]["label"][:2000], dtype=np.float32)

vectorizer = TfidfVectorizer(max_features=12000, ngram_range=(1, 2), min_df=3)
X_train = vectorizer.fit_transform(train_text).astype(np.float32).toarray()
X_test = vectorizer.transform(test_text).astype(np.float32).toarray()

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train), torch.tensor(train_label).view(-1, 1)),
    batch_size=64,
    shuffle=True,
)
test_loader = DataLoader(
    TensorDataset(torch.tensor(X_test), torch.tensor(test_label).view(-1, 1)),
    batch_size=256,
)

print("Train features:", X_train.shape)
print("Test features:", X_test.shape)


In [ ]:
swish_expr = Expression("swish", "x / (1 + exp(-x))")
swish = to_pytorch(swish_expr)

class SentimentNet(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 256)
        self.act = swish
        self.fc2 = nn.Linear(256, 64)
        self.fc3 = nn.Linear(64, 1)

    def forward(self, x):
        h1 = self.act(self.fc1(x))
        h2 = torch.relu(self.fc2(h1))
        logits = self.fc3(h2)
        return logits, h2

model = SentimentNet(X_train.shape[1]).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

base_logger = TrainingLogger(level=LogLevel.EXPERT)
storage = TieredStorage(base_dir="./logs_imdb_sentiment", write_debug=True)
dashboard = DashboardExporter(path="./logs_imdb_sentiment/dashboard.html")
base_logger.add_backend(storage)
base_logger.add_backend(dashboard)

logger = AdaptiveLogger(base_logger, config=AnomalyConfig(gradient_spike_factor=4.0, loss_spike_pct=30.0))
warnings_engine = AutoHealthWarnings()

epochs = 4
logger.on_train_start(total_epochs=epochs, total_samples=len(train_loader.dataset), batch_size=64)

train_loss_hist = []
val_loss_hist = []

for epoch in range(epochs):
    logger.on_epoch_start(epoch)
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for batch_idx, (xb, yb) in enumerate(train_loader):
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits, hidden = model(xb)
        loss = criterion(logits, yb)
        loss.backward()

        grad_norms = {}
        for n, p in model.named_parameters():
            if p.grad is not None:
                gn = float(p.grad.data.norm().item())
                grad_norms[n] = gn
                logger.on_gradient_computed(n, gn)

        optimizer.step()

        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()
        acc = float((preds == yb).float().mean().item())
        batch_loss = float(loss.item())

        hidden_np = hidden.detach().cpu().numpy()
        alerts = warnings_engine.check_batch(
            epoch=epoch,
            batch=batch_idx,
            loss=batch_loss,
            gradient_norms=grad_norms,
            activation_stats={
                "hidden": {
                    "activation_type": "relu",
                    "zeros_pct": float((np.abs(hidden_np) < 1e-8).mean() * 100),
                    "saturation_pct": float((np.abs(hidden_np) > 0.99).mean() * 100),
                }
            },
        )
        for alert in alerts:
            logger.on_health_check(
                check_name=alert.rule_name,
                severity=alert.severity,
                message=alert.message,
                recommendations=alert.recommendations,
            )

        logger.on_batch_end(batch_idx, epoch=epoch, loss=batch_loss, metrics={"accuracy": acc})
        running_loss += batch_loss * xb.size(0)
        running_correct += int((preds == yb).sum().item())
        running_total += yb.numel()

    train_loss = running_loss / len(train_loader.dataset)
    train_acc = running_correct / running_total

    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits, _ = model(xb)
            vloss = criterion(logits, yb)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()
            val_loss_sum += float(vloss.item()) * xb.size(0)
            val_correct += int((preds == yb).sum().item())
            val_total += yb.numel()

    val_loss = val_loss_sum / len(test_loader.dataset)
    val_acc = val_correct / val_total

    train_loss_hist.append(train_loss)
    val_loss_hist.append(val_loss)

    logger.on_epoch_end(
        epoch,
        metrics={
            "loss": train_loss,
            "accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
        },
    )

logger.on_train_end(final_metrics={"loss": train_loss_hist[-1], "val_loss": val_loss_hist[-1]})
dashboard_path = dashboard.save()
storage.flush()
storage.close()

print("Anomaly summary:", logger.get_anomaly_summary())
print("Dashboard:", dashboard_path)


In [ ]:
model.eval()
all_preds = []
all_true = []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits, _ = model(xb)
        preds = (torch.sigmoid(logits).cpu().numpy().reshape(-1) > 0.5).astype(int)
        all_preds.extend(preds.tolist())
        all_true.extend(yb.numpy().reshape(-1).astype(int).tolist())

print(classification_report(all_true, all_preds, digits=3))

plt.figure(figsize=(8, 4))
plt.plot(train_loss_hist, label="train_loss")
plt.plot(val_loss_hist, label="val_loss")
plt.title("IMDb Sentiment Classifier")
plt.legend()
plt.show()
